# Notebook 07 - Second-order approximation (R6)

Faithful port of `Second_Order_Simulation.m`.

Computes a second-order Taylor approximation of log GDP around the
baseline (A = 1) using finite-difference Hessians.

**Two cases:**
1. **No-reallocation (fixed labor):** Hessian via finite differences of nominal GDP
2. **Full-reallocation (mobile labor, w=1):** Hessian via finite differences of Domar weights

The approximation formula:
$$
\log \text{GDP}(A) \approx \boldsymbol{\lambda}' \cdot \log(A) +
\frac{1}{2} \cdot \log(A)' \cdot \mathbf{H} \cdot \log(A)
$$
where $\boldsymbol{\lambda}$ are Domar weights and $\mathbf{H}$ is the Hessian-type matrix.

In [2]:
using LinearAlgebra, Statistics, Printf, DelimitedFiles

const NB_DIR = @__DIR__
const PKG = joinpath(NB_DIR, "..")
const DATA_DIR = joinpath(PKG, "..", "Replication Files", "GDP Simulatin -- 88 Sector")
const RESULTS_DIR = joinpath(PKG, "data", "results")
mkpath(RESULTS_DIR)
push!(LOAD_PATH, joinpath(PKG, "src"))
include(joinpath(PKG, "src", "BFReplication.jl"))
using .BFReplication
using .BFReplication.DataLoader

println("Module loaded OK")

Module loaded OK


In [3]:
data = load_bf_data(joinpath(DATA_DIR, "BFdata.csv"); year=1980)
stfp, _, _ = load_tfp_data(joinpath(DATA_DIR, "stfp.csv"))
Sigma_yearly, Sigma_4year = empirical_covariances(stfp)
Cov4 = Matrix(Diagonal(diag(Sigma_4year)))

λ = (I - Diagonal(1 .- data.α) * data.Ω)' \ data.β
println("N sectors = $(data.N), sum(λ) = $(round(sum(λ), digits=4))")

N sectors = 76, sum(λ) = 2.0918


## No-reallocation (fixed labor) Hessian

Each column of H is constant: H[:,i] = (GDP(A + h·e_i) - GDP(A)) / h

This gives a rank-1 matrix (all rows identical per column).

In [4]:
# WARNING: full 76-sector Hessian is slow (76 equilibrium solves)
# Test with a few sectors first
println("No-realloc Hessian (first 5 sectors):")
H_nr = second_order_hessian_norealloc(data.Ω, data.α, data.β, data.L, 0.5, 0.001, 0.9; h=1e-6)
for i in 1:5
    println("  $(round.(H_nr[i, 1:5], digits=6))")
end
println("  rank = $(rank(H_nr[1:5,1:5])) — degenerate")

No-realloc Hessian (first 5 sectors):
  [0.081816, 0.0043, 0.003978, 0.00466, 0.004317]
  [0.081816, 0.0043, 0.003978, 0.00466, 0.004317]
  [0.081816, 0.0043, 0.003978, 0.00466, 0.004317]
  [0.081816, 0.0043, 0.003978, 0.00466, 0.004317]
  [0.081816, 0.0043, 0.003978, 0.00466, 0.004317]
  rank = 1 — degenerate


## Full-reallocation (mobile labor) Hessian

For each sector i, perturb A_i by h, compute new Domar weights
λ_new = p · y / C, and set H[:,i] = (λ_new - λ) / h.

This gives a proper N×N matrix (negative diagonal, small off-diagonals).

In [5]:
# WARNING: full 76-sector Hessian takes ~76 equilibrium solves (~5-10 min)
# Test with a few sectors
println("Realloc Hessian (first 5 sectors, ε=θ=σ=0.5/0.001/0.9):")
H_r = second_order_hessian_realloc(data.Ω, data.α, data.β, 0.5, 0.001, 0.9; h=1e-5)
for i in 1:5
    println("  $(round.(H_r[i, 1:5], digits=6))")
end
println("  rank = $(rank(H_r[1:5,1:5])) — proper N×N")

Realloc Hessian (first 5 sectors, ε=θ=σ=0.5/0.001/0.9):
  [-0.083425, -0.001613, 0.000336, 0.000122, 4.1e-5]
  [-0.001613, -0.004117, -0.000414, 1.6e-5, 2.0e-5]
  [0.000336, -0.000414, -0.004028, 2.3e-5, 2.8e-5]
  [0.000122, 1.6e-5, 2.3e-5, -0.005669, 2.7e-5]
  [4.1e-5, 2.0e-5, 2.8e-5, 2.7e-5, -0.003931]
  rank = 5 — proper N×N


## Second-order MC (realloc Hessian, small test)

In [6]:
# Quick MC with 5-sector Hessian
println("5-sector second-order MC (200 draws, 4-yr cov):")
mc5 = second_order_mc(λ[1:5], H_r[1:5, 1:5], Cov4[1:5, 1:5]; trials=200, seed=42)
println("  mean   = $(round(mc5.mean*100, digits=4)) %")
println("  std    = $(round(mc5.std*100, digits=4)) %")
println("  skew   = $(round(mc5.skewness, digits=3))")
println("  exkurt = $(round(mc5.excess_kurtosis, digits=3))")

5-sector second-order MC (200 draws, 4-yr cov):
  mean   = -0.0196 %
  std    = 0.4007 %
  skew   = -0.055
  exkurt = 0.362


## Full run (Mac recommended)

The following runs the full 76-sector Hessian and 50k-draw MC.
It runs in ~2-3 sec on a Mac (only 76 cheap equilibrium solves for the Hessian; MC is matrix ops).

```julia
# Full realloc
r = run_second_order_realloc(data, Cov4; trials=50000, seed=42)
println("mean=$(r.mc.mean) std=$(r.mc.std) skew=$(r.mc.skewness) exkurt=$(r.mc.excess_kurtosis)")

# Full no-realloc (faster — no equilibrium solves needed after Hessian)
mc_nr = run_second_order_norealloc(data, Cov4; trials=50000, seed=42)
println("mean=$(mc_nr.mc.mean) std=$(mc_nr.mc.std) skew=$(mc_nr.mc.skewness) exkurt=$(mc_nr.mc.excess_kurtosis)")
```

In [8]:
# Full realloc
r = run_second_order_realloc(data, Cov4; trials=50000, seed=42)
println("mean=$(r.mc.mean) std=$(r.mc.std) skew=$(r.mc.skewness) exkurt=$(r.mc.excess_kurtosis)")

# Full no-realloc (faster — no equilibrium solves needed after Hessian)
mc_nr = run_second_order_norealloc(data, Cov4; trials=50000, seed=42)
println("mean=$(mc_nr.mc.mean) std=$(mc_nr.mc.std) skew=$(mc_nr.mc.skewness) exkurt=$(mc_nr.mc.excess_kurtosis)")

mean=-0.010416537739686468 std=0.024372545094793075 skew=-0.1839675140233203 exkurt=0.0826558272838045
mean=0.0007992424361113842 std=0.022136775509341233 skew=1.3360089623716773 exkurt=3.4922864884550755


In [9]:
mean(diag(Sigma_yearly)) 

0.0014026936399479588